# ML-07 — Baseline Action Score and Top-10 Review

**Lane: content refresh prioritization** — *"which pages should my content team refresh first?"*
Fill the rule, prove the signals it leans on, rank all 30k starter pages, and read the top ten with a skeptic's eye. This rule is the baseline my Week-5 model must beat.

Everything here runs on the bundled starter file `data/raw/content_refresh_anonymized.csv` (30,000 rows, one per content page, trailing-90-day metrics). No warehouse queries, no future windows.

## 1. My rule, the signals it leans on, and its reason codes

**The rule in plain words (decided first):**

> Refresh the page that still gets real search traffic but is slipping — either it's gone stale
> (not updated in 6+ months) or its title/meta is not earning the clicks its position deserves
> (a top-10 page with a very low CTR). Order the whole queue by **how much traffic is at stake**.

Before coding it, I checked the **two signals the rule leans on**, with a bucket table and an `n` on every row. Both are signals behind real FlyRank flags from the session:

1. **Staleness** → behind the *refresh* flags ("page is old, refresh it").
2. **CTR-vs-position** → behind the *CTR-fix* logic ("good position, no clicks → fix the title/meta").

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

# --- find the repo root, wherever the kernel starts ---
ROOT = Path(os.getcwd()).resolve()
for _ in range(6):
    if (ROOT / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        break
    ROOT = ROOT.parent

CSV_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
OUT_DIR = ROOT / "work" / "outputs"
OUT_CSV = OUT_DIR / "baseline_action_score.csv"
OUT_JSON = OUT_DIR / "baseline_action_metrics.json"

df = pd.read_csv(CSV_PATH)

# Evaluation label ONLY. Per the data dictionary, is_declining_label == (trend_direction == "down")
# and trend_direction is computed from trend_pct. So NEITHER trend column may ever be a rule input.
# The label below is used for measuring the queue (precision@K) and nothing else.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(f"Repo root : {ROOT}")
print(f"Rows      : {len(df):,}  (one row per content page)")
print(f"Columns   : {df.shape[1]}")
print(f"Base declining rate (the rate random picks would hit): {df['is_declining_label'].mean():.3f}")

Repo root : /Users/jh/ML_Flyrank_Ai_Assignment
Rows      : 30,000  (one row per content page)
Columns   : 45
Base declining rate (the rate random picks would hit): 0.542


### Signal check 1 — staleness behind the refresh flags

**Claim:** "pages not updated in a long time are more likely to be in decline" → that's why the
refresh flags fire on old pages.

**Test:** declining rate by `days_since_last_update` bucket, `n` on every row, plus a quick look at
whether 'old' is really per-page neglect or a handful of bulk-update dates.

In [2]:
# --- Signal check 1: staleness behind the refresh flags ---
bucket = pd.cut(
    df["days_since_last_update"],
    bins=[0, 30, 90, 180, 1_000_000],
    right=True,
    labels=["0-30", "31-90", "91-180", "181+"],
)
sig1 = (
    df.groupby(bucket, observed=True)
    .agg(n=("content_id", "size"), decl_rate=("is_declining_label", "mean"), median_impressions=("impressions_90d", "median"))
    .assign(decl_rate=lambda t: (t["decl_rate"] * 100).round(1))
)
print("Declining rate by days since last update  (n on every row):")
print(sig1)

print("\nMost common 'last update day' values (clumped dates = bulk/scripted refreshes):")
print(df["days_since_last_update"].value_counts().head(5).to_string())
print(f"\nRows sharing the SAME last-update day as the most common 5 dates: "
      f"{(df['days_since_last_update'].isin(df['days_since_last_update'].value_counts().head(5).index)).mean()*100:.0f}% of all pages")

Declining rate by days since last update  (n on every row):
                            n  decl_rate  median_impressions
days_since_last_update                                      
0-30                    20480       51.1               470.0
31-90                     175       58.9               510.0
91-180                   9171       61.1              1692.0
181+                      174       47.1                15.5

Most common 'last update day' values (clumped dates = bulk/scripted refreshes):
days_since_last_update
20     11573
104     8773
22      3564
8       1929
13       515

Rows sharing the SAME last-update day as the most common 5 dates: 88% of all pages


**Verdict: MIXED.** Decline rises with staleness through 91–180 days (61.1%, n=9,171 vs 51.1%,
n=20,480 freshly-updated) — the refresh flag has real support in that range. But the trend is *not*
monotonic: the most-stale bucket (181+ days, n=174) *reverses* to 47.1%, and those pages are mostly
already dead (median 15.5 impressions in 90 days — no traffic left to lose). Worse, 30%+ of pages
share just five "last update" dates (20, 22, 104, 8 days…), so "old" often means "a bulk refresh
campaign ran long ago," not "editors neglected this specific page." The signal survives but needs a
volume floor and a lot of humility — exactly why my rule treats staleness as a *condition*, never
as the whole story.

### Signal check 2 — CTR-vs-position behind the CTR-fix logic

**Claim:** "a page ranking on page 1 should be earning clicks; a top-10 page with very low CTR is
underperforming and needs a title/meta fix."

**Test:** CTR by position tier among *visible* pages (position > 0 and impressions ≥ 300 — CTR of a
few impressions is noise), n on every row. Rates are clicks-weighted where possible (rates need
denominators). Then the decision-relevant split: among top-10 pages, do low-CTR pages decline more?

In [3]:
# --- Signal check 2: CTR vs position behind the CTR-fix logic ---
vis = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 300)]
print(f"Visible pages used for this test (position > 0 and impressions >= 300): {len(vis):,}")

sig2 = (
    vis.groupby("position_tier", observed=True)
    .agg(
        n=("content_id", "size"),
        p25_ctr=("ctr", lambda x: x.quantile(0.25)),
        p50_ctr=("ctr", lambda x: x.quantile(0.50)),
        p75_ctr=("ctr", lambda x: x.quantile(0.75)),
        mean_ctr=("ctr", "mean"),
        decl_rate=("is_declining_label", "mean"),
    )
    .assign(
        p25_ctr=lambda t: (t["p25_ctr"] * 100).round(2),
        p50_ctr=lambda t: (t["p50_ctr"] * 100).round(2),
        p75_ctr=lambda t: (t["p75_ctr"] * 100).round(2),
        mean_ctr=lambda t: (t["mean_ctr"] * 100).round(2),
        decl_rate=lambda t: (t["decl_rate"] * 100).round(1),
    )
)
print("\nCTR (%, whole 90-day window) by average position tier, n on every row:")
print(sig2)

cw = (
    vis.groupby("position_tier", observed=True)
    .agg(n=("content_id", "size"), clicks=("clicks_90d", "sum"), impressions=("impressions_90d", "sum"))
)
cw["wctr_pct"] = (100 * cw["clicks"] / cw["impressions"]).round(2)
print("\nClicks-weighted CTR (%) by position tier (clicks across all pages / impressions across all pages):")
print(cw)

sp = vis[["avg_position", "ctr"]].corr(method="spearman").iloc[0, 1]
print(f"\nSpearman corr(avg_position, ctr) = {sp:.3f}  (negative = better position -> higher CTR)")

top10 = vis[vis["avg_position"] <= 10]
low = (top10["ctr"] < 0.5).astype(int)
split = (
    top10.assign(low_ctr=low)
    .groupby("low_ctr", observed=True)
    .agg(n=("content_id", "size"), decl_rate=("is_declining_label", "mean"))
    .assign(low_ctr=lambda t: t.index.map({0: "CTR >= 0.5%", 1: "CTR < 0.5%"}), decl_rate=lambda t: (t["decl_rate"] * 100).round(1))
    .reset_index(drop=True)
)
print("\nDeclining rate among top-10 visible pages, split by low CTR (< 0.5%):")
print(split)

Visible pages used for this test (position > 0 and impressions >= 300): 18,752

CTR (%, whole 90-day window) by average position tier, n on every row:
                  n  p25_ctr  p50_ctr  p75_ctr  mean_ctr  decl_rate
position_tier                                                      
deep            562      0.0      0.0      2.0      4.34       29.5
page_1         7623     11.0     23.0     44.0     33.77       59.1
page_3_5       5004      0.0      8.0     20.0     14.03       59.6
striking       5078      6.0     17.0     33.0     25.98       61.7
top_3           485      6.0     20.0     48.0     33.58       75.3

Clicks-weighted CTR (%) by position tier (clicks across all pages / impressions across all pages):
                  n  clicks  impressions  wctr_pct
position_tier                                     
deep            562     431      1154517      0.04
page_1         7623  312263     89306572      0.35
page_3_5       5004   54066     34937984      0.15
striking       507

**Verdict: CONFIRMED.** The CTR-position curve is real in this data: clicks-weighted CTR rises
from 0.04% in deep positions to 0.49% at top-3, and Spearman(position, CTR) = −0.37 across 18,752
visible pages. And crucially for the fix logic — among top-10 pages, low-CTR pages (< 0.5%) decline
**63.7%** (n=6,436) vs **46.5%** (n=1,718) for the rest. A top-10 page with a low CTR is ~17 points
more likely to be in decline. That is the signal my rule leans on hardest.

### The rule, encoded: score → ONE reason code → one action

**Plain words:** *refresh the still-trafficked page that is stale and/or underperforming its
position on CTR; put the big-traffic pages first.*

```python
stale   = (days_since_last_update >= 180) and (impressions_90d >= 300)
ctr_gap = (0 < avg_position <= 10) and (ctr < 0.5) and (impressions_90d >= 300)

score   = log1p(impressions_90d) * (1 + stale) * (1 + ctr_gap)   # readable, no fitted weights
```

Every page gets **exactly one** reason code (highest-priority flag that fires) and **one** action label:

| reason code | condition | action label |
|---|---|---|
| `stale_and_low_ctr_visible` | stale AND ctr-gap | `refresh_and_rewrite_title_meta` |
| `stale_visible` | stale only | `refresh_content` |
| `low_ctr_good_position` | ctr-gap only | `rewrite_title_meta` |
| `monitor` | neither | `monitor` |

In [4]:
# --- The rule in code ---
impressions = df["impressions_90d"]

stale_flag = ((df["days_since_last_update"] >= 180) & (impressions >= 300)).astype(int)
ctr_gap_flag = (
    (df["avg_position"] > 0)
    & (df["avg_position"] <= 10)
    & (df["ctr"] < 0.5)
    & (impressions >= 300)
).astype(int)

# Transparent score: volume drives rank; each live flag doubles the weight of that page.
df["score"] = np.log1p(impressions) * (1 + stale_flag) * (1 + ctr_gap_flag)

REASON = {
    (1, 1): "stale_and_low_ctr_visible",
    (1, 0): "stale_visible",
    (0, 1): "low_ctr_good_position",
    (0, 0): "monitor",
}
ACTION = {
    "stale_and_low_ctr_visible": "refresh_and_rewrite_title_meta",
    "stale_visible": "refresh_content",
    "low_ctr_good_position": "rewrite_title_meta",
    "monitor": "monitor",
}

df["reason_code"] = [REASON[(s, g)] for s, g in zip(stale_flag, ctr_gap_flag)]
df["action"] = df["reason_code"].map(ACTION)

print(df.groupby("reason_code", observed=True).size().to_string())
print("\nThe rule uses NO label-derived column (the label is for evaluation only) and NO future window:",
      "\n  inputs are impressions_90d, days_since_last_update, avg_position, ctr — all from the trailing-90-day snapshot.")

reason_code
low_ctr_good_position         6432
monitor                      23546
stale_and_low_ctr_visible        4
stale_visible                   18

The rule uses NO label-derived column (the label is for evaluation only) and NO future window: 
  inputs are impressions_90d, days_since_last_update, avg_position, ctr — all from the trailing-90-day snapshot.


## 2. Build the ranked queue → writes `work/outputs/baseline_action_score.csv`

Rank all 30,000 pages, write the CSV from the notebook, and print the honest evaluation:
precision@K against the base rate. The CSV is regenerated every run and stays out of git by design
(`.gitignore` blocks `work/**/*.csv`); the metrics JSON is the committed receipt.

In [5]:
# --- 2. Rank everything and write the queue ---
df["rank"] = df["score"].rank(method="first", ascending=False).astype(int)
queue = df.sort_values("rank")

OUT_DIR.mkdir(parents=True, exist_ok=True)
queue[
    ["content_id", "client_id", "rank", "score", "reason_code", "action",
     "impressions_90d", "clicks_90d", "ctr", "avg_position", "days_since_last_update",
     "is_declining_label"]
].to_csv(OUT_CSV, index=False)

base_rate = df["is_declining_label"].mean()
p_at = {k: round(float(queue.head(k)["is_declining_label"].mean()), 3) for k in (5, 10, 20, 50, 100)}

print(f"Wrote ranked queue: {OUT_CSV.relative_to(ROOT)}  ({len(queue):,} rows)")
print(f"Base declining rate: {base_rate:.3f}")
for k in (5, 10, 20, 50, 100):
    print(f"  Precision@{k:<3}: {p_at[k]:.3f}")

print("\nQueue composition by reason code:")
print(
    queue.groupby("reason_code", observed=True)
    .agg(n=("content_id", "size"), decl_rate=("is_declining_label", "mean"), median_impressions=("impressions_90d", "median"))
    .assign(decl_rate=lambda t: (t["decl_rate"] * 100).round(1))
)

metrics = {
    "notebook": "w04_baseline_score.ipynb",
    "lane": "content_refresh_prioritization",
    "n_rows": int(len(queue)),
    "base_rate": round(base_rate, 3),
    "precision_at_k": p_at,
    "reason_codes": {k: int(v) for k, v in queue["reason_code"].value_counts().items()},
    "score_formula": "log1p(impressions_90d) * (1 + stale_flag) * (1 + ctr_gap_flag)",
    "rule": "stale_flag  = days_since_last_update >= 180 and impressions_90d >= 300; "
            "ctr_gap_flag = 0 < avg_position <= 10 and ctr < 0.5 and impressions_90d >= 300",
    "rule_inputs": ["impressions_90d", "days_since_last_update", "avg_position", "ctr"],
    "evaluation_only": ["is_declining_label"],
}
OUT_JSON.write_text(json.dumps(metrics, indent=2, sort_keys=True))
print(f"\nWrote metrics receipt: {OUT_JSON.relative_to(ROOT)}")

Wrote ranked queue: work/outputs/baseline_action_score.csv  (30,000 rows)
Base declining rate: 0.542
  Precision@5  : 0.800
  Precision@10 : 0.800
  Precision@20 : 0.550
  Precision@50 : 0.480
  Precision@100: 0.450

Queue composition by reason code:
                               n  decl_rate  median_impressions
reason_code                                                    
low_ctr_good_position       6432       63.7              3738.0
monitor                    23546       51.6               359.0
stale_and_low_ctr_visible      4      100.0               887.5
stale_visible                 18       77.8              3063.0

Wrote metrics receipt: work/outputs/baseline_action_metrics.json


## 3. Top-10 review — read my own queue with a skeptic's eye

The queue's top 10 (next cell prints it); then one line per page below: the action, why it's there,
and what would make it wrong.

In [6]:
# --- 3. Show the top 10 for the manual review ---
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 230)
top10 = queue.head(10).copy()
top10["ctr_pct"] = top10["ctr"].round(2)
top10["imp_k"] = (top10["impressions_90d"] / 1000).round(1)
show = top10[[
    "rank", "reason_code", "action", "imp_k", "clicks_90d",
    "days_since_last_update", "avg_position", "ctr_pct", "is_declining_label",
    "content_type", "main_intent", "content_id",
]].rename(columns={
    "imp_k": "impressions_000s", "avg_position": "avg_pos", "ctr_pct": "ctr_%",
    "is_declining_label": "declining?", "days_since_last_update": "days_since_update",
})
print("Top 10 (CTR is a %, avg_pos 1 = best, declining? = evaluation label shown for context only):")
print(show.to_string(index=False))

Top 10 (CTR is a %, avg_pos 1 = best, declining? = evaluation label shown for context only):
 rank               reason_code                         action  impressions_000s  clicks_90d  days_since_update  avg_pos  ctr_%  declining?    content_type   main_intent           content_id
    1 stale_and_low_ctr_visible refresh_and_rewrite_title_meta               1.4           4                183      7.8   0.28           1 keyword article informational content_e3ff1b093148
    2 stale_and_low_ctr_visible refresh_and_rewrite_title_meta               1.0           4                301      9.0   0.42           1 keyword article           NaN content_7f116ae1f6f5
    3 stale_and_low_ctr_visible refresh_and_rewrite_title_meta               0.8           2                301      5.8   0.24           1 keyword article           NaN content_72496874f806
    4     low_ctr_good_position             rewrite_title_meta             517.7         741                104      4.2   0.14           1 key

**Per-page review — action · why it's here · what would make it wrong**

1. **content_e3ff1b093148** — `refresh_and_rewrite_title_meta` · only page in the top ten where BOTH flags fire — truly stale (183 days since update) *and* low CTR at position ~8, so the double-weight pushed it above bigger pages · **wrong if**: its CTR "0.28%" rests on just **4 clicks** — coin-flip noise, and 1.4k impressions mean the refresh payoff is tiny next to the 500k-impression rows below it.
2. **content_7f116ae1f6f5** — `refresh_and_rewrite_title_meta` · both flags again on a genuinely ancient page (301 days), declining · **wrong if**: 4 clicks again; a 954-impression page 10 months stale may be beyond revival — a full rewrite, not a refresh, and only if the query still has demand.
3. **content_72496874f806** — `refresh_and_rewrite_title_meta` · both flags, 301 days stale, declining · **wrong if**: 2 clicks — its CTR is a coin flip, and flagging a third tiny page at #3 shows the double-weight loves small stale pages; the queue may under-weight sheer volume.
4. **content_5fe46e04994d** — `rewrite_title_meta` · 517k impressions, position 4.2, CTR 0.14% on a *declining* page — the canonical "huge visibility, title not earning clicks" fix · **wrong if**: its 104-day "staleness" is a bulk-update artifact (30% of pages share 5 update dates), the CTR floor may be query-mix–driven, and rewriting could disturb a page-1 asset.
5. **content_aaef01a50def** — `rewrite_title_meta` · 517k impressions, position 5.4, CTR 0.25% — pure volume × ctr-gap, no staleness (updated 22 days ago) · **wrong if**: it is **not currently declining** (label 0) — urgency is much lower than its rank suggests, and a rewrite 3 weeks after the last update may collide with it.
6. **content_8c19996aa890** — `rewrite_title_meta` · position 2.5 with CTR 0.15% on half a million impressions, declining — strongest pure fix case in the queue · **wrong if**: its impressions come from low-intent queries where 0.15% is the norm; or the title already wins head terms and the decline is algorithmic/seasonal, not title-driven.
7. **content_4c36c775b818** — `rewrite_title_meta` · position 2.3, CTR 0.41%, declining — best CTR of the giants but still far under a top-3 norm · **wrong if**: 0.41% is near this query-mix's ceiling (a smaller relative gap than the score implies) and "declining" here is driven by impressions, not clicks.
8. **content_1a9e894be2e2** — `rewrite_title_meta` · 416k impressions, position 4.0, CTR 0.23%, declining, transactional intent · **wrong if**: transactional pages earn lower CTR by nature, and a title rewrite may not move the clicks it does get — this may be a landing-page problem instead.
9. **content_db5989a78dd3** — `rewrite_title_meta` · 345k impressions, position 5.4, CTR 0.21% · **wrong if**: **not declining** (label 0); commercial intent pages may need page content, not a title tweak, and CTR<0.5% flags half the visible population — it's a broad net, not a rare diagnosis.
10. **content_cb112fce36be** — `rewrite_title_meta` · 310k impressions, position 5.6, CTR 0.16%, declining · **wrong if**: 492 clicks still make the CTR estimate noisy; the 104-day bulk cohort again clouds the story; average position 5.6 may average over spikes, diluting true CTR.

Pattern the review exposes: **rows 1–3 are "true" the way I hoped, rows 4, 6–8, 10 are the
highest-confidence fixes, and rows 5 and 9 are volume-and-CTR calls on pages not even currently
declining** — the honest cost of an action rule that doesn't peek at the label.

## 4. Weak picks + leakage check

**Weakest picks in the top 10:** rows 1–3 (their low-CTR flag rests on 2–4 clicks; their volumes are
1–1.4k impressions — smallest payoffs in the top ten) and rows 5 & 9 (labelled *not* declining, so
urgency is lower than rank implies). Below, the numbers behind those worries, then the leakage check.

In [7]:
# --- 4. Weak picks: quantify the shaky edges ---
top10 = queue.head(10)
print("Clicks behind the low-CTR judgement in the top 10 (tiny counts = noisy rates):")
print(top10[["rank", "impressions_90d", "clicks_90d", "ctr", "reason_code"]].to_string(index=False))
print(f"\nTop-10 pages backed by <= 5 clicks: {int((top10['clicks_90d'] <= 5).sum())} of 10")

print("\n'Last update' clumps onto a few dates (bulk/scripted refreshes), so staleness is partly an artifact:")
print(df["days_since_last_update"].value_counts().head(5).to_string())

print(f"\nThe ctr-gap flag is a BROAD net: {(df['reason_code'] != 'monitor').sum():,} of {len(df):,} pages flagged "
      f"({(df['reason_code'] != 'monitor').mean()*100:.0f}%), and rows 5/9 in the top 10 are not even declining.")

print("\nPrecision decays fast below the very top (the honest target for the Week-5 model to beat):")
for k in (5, 10, 20, 50, 100):
    print(f"  P@{k:<3} = {queue.head(k)['is_declining_label'].mean():.3f}   (base rate {base_rate:.3f})")

Clicks behind the low-CTR judgement in the top 10 (tiny counts = noisy rates):
 rank  impressions_90d  clicks_90d  ctr               reason_code
    1             1408           4 0.28 stale_and_low_ctr_visible
    2              954           4 0.42 stale_and_low_ctr_visible
    3              821           2 0.24 stale_and_low_ctr_visible
    4           517715         741 0.14     low_ctr_good_position
    5           517109        1270 0.25     low_ctr_good_position
    6           509252         785 0.15     low_ctr_good_position
    7           463103        1889 0.41     low_ctr_good_position
    8           416180         944 0.23     low_ctr_good_position
    9           345111         733 0.21     low_ctr_good_position
   10           309910         492 0.16     low_ctr_good_position

Top-10 pages backed by <= 5 clicks: 3 of 10

'Last update' clumps onto a few dates (bulk/scripted refreshes), so staleness is partly an artifact:
days_since_last_update
20     11573
104     8773

**Leakage check — confirmed clean:**

- **Rule inputs:** `impressions_90d`, `days_since_last_update`, `avg_position`, `ctr`. All four are
  trailing-90-day snapshot columns measured *at decision time*. No date partitioning and no future
  window exists anywhere — the source is a single export file.
- **Label-derived inputs:** `is_declining_label` is computed from `trend_direction`/`trend_pct` per
  the data dictionary; it appears in the queue **only as the evaluation column** and never enters the
  score. `trend_direction` and `trend_pct` are never used as features.
- **Output hygiene:** the CSV is regenerated on every run and is blocked from git by design
  (`work/**/*.csv` in `.gitignore`). The committed receipt is `work/outputs/baseline_action_metrics.json`.
- **Honest claim wording:** these are measured relationships in one 30k-row snapshot, decision-support
  only. "CONFIRMED" for CTR-vs-position, "MIXED" for staleness, and a top-10 precision of **0.80 vs a
  0.542 base rate** are the numbers a model must beat — not proof about the real world.

## Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] Two signal checks with visible bucket tables and n; at least one flag-linked (both are)
- [x] One rule: a readable score, ONE reason code per row, one action label
- [x] Ranked queue written from the notebook to `work/outputs/baseline_action_score.csv`
- [x] Ten reviewed rows, each with action, why it's there, and what would make it wrong
- [x] No client names/URLs; no future-window or label-derived inputs to the rule
- [x] Committed under `work/notebooks/` — repo URL submitted on the card. Done.